In [2]:
import torch
import torch.nn as nn

In [8]:
def get_sinusoidal_embeddings(seq_len, d_model):
  # out -> [seq_len, d_model]

  positions = torch.arange(0, seq_len).unsqueeze(1) # [seq_len]

  dims = torch.arange(0, d_model, 2).unsqueeze(0) # [d_model//2]

  mian = torch.exp(torch.log(torch.tensor(10000, dtype=torch.float32))*(dims/d_model))

  angles = positions/mian

  #print(final.shape) [50, 128]
  w, _ = angles.shape

  #print(w)
  #print(h)

  final = torch.ones(w, d_model)

  final[:, 0::2] = torch.sin(angles)
  final[:, 1::2] = torch.cos(angles)

  return final

#print(get_sinusoidal_embeddings(50, 256))

In [3]:
class CrossAttention(nn.Module):
  def __init__(self, model_dim, num_heads=6):
    super().__init__()
    self.WQ = nn.Linear(model_dim, model_dim)
    self.WK = nn.Linear(model_dim, model_dim)
    self.WV = nn.Linear(model_dim, model_dim)
    self.WO = nn.Linear(model_dim, model_dim)
    self.num_heads = num_heads
    self.head_dim = model_dim // num_heads

    assert model_dim % num_heads == 0

  def forward(self, x_dec, x_enc):
    # x_dec -> [B, S_dec, D]
    # x_enc -> [B, S_enc, D]
    B_d, L_d, _ = x_dec.shape
    B_e, L_e, _ = x_enc.shape

    Q = self.WQ(x_dec) # [B, L_d, model_dim]
    Q = torch.reshape(Q, (B_d, L_d, self.num_heads, self.head_dim)) # [B, L_d, num_heads, heads_dim]
    Q = torch.permute(Q, (0, 2, 1, 3)) # [B, num_heads, L_d, heads_dim]

    K = self.WK(x_enc)
    K = torch.reshape(K, (B_e, L_e, self.num_heads, self.head_dim))
    K = torch.permute(K, (0, 2, 1, 3)) # [B, num_heads, L_e, heads_dim]

    V = self.WV(x_enc)
    V = torch.reshape(V, (B_e, L_e, self.num_heads, self.head_dim))
    V = torch.permute(V, (0, 2, 1, 3))

    scores = torch.matmul(Q, K.transpose(-2, -1)) # [B, num_heads, L_d, L_e]

    scores = scores / math.sqrt(self.head_dim)

    #mask = torch.triu(torch.ones(L, L), diagonal=1).bool()

    #scores = scores.masked_fill(mask, float('-inf'))

    out = torch.softmax(scores, dim=-1)

    final = torch.matmul(out, V) # [B, num_heads, L_d, heads_dim]

    final = torch.permute(final, (0, 2, 1, 3)) # [B, L_d, num_heads, heads_dim]

    final = torch.reshape(final, (B_d, L_d, self.num_heads*self.head_dim)) #[B, L_d, model_dim]

    return self.WO(final)



In [4]:
class SelfAttention(nn.Module):
  def __init__(self, model_dim, num_heads=6):
    super().__init__()
    self.WQ = nn.Linear(model_dim, model_dim) # num_heads * head_dim = model_dim
    self.WK = nn.Linear(model_dim, model_dim)
    self.WV = nn.Linear(model_dim, model_dim)
    self.WO = nn.Linear(model_dim, model_dim)
    self.model_dim = model_dim
    self.num_heads = num_heads
    self.head_dim = model_dim // num_heads

    assert model_dim % num_heads == 0

  def forward(self, q, k, v, casual=False):
    B, Lq, _ = q.shape
    _, Lk, _ = k.shape
    _, Lv, _ = v.shape

    Q = self.WQ(q) # [B, Seq_len, model_dim]
    Q = torch.reshape(Q, (B, Lq, self.num_heads, self.head_dim)) # [B, Seq_len, num_heads, heads_dim]
    Q = torch.permute(Q, (0, 2, 1, 3)) # [B, num_heads, seq_len, heads_dim]

    K = self.WK(k)
    K = torch.reshape(K, (B, Lk, self.num_heads, self.head_dim))
    K = torch.permute(K, (0, 2, 1, 3))

    V = self.WV(v)
    V = torch.reshape(V, (B, Lv, self.num_heads, self.head_dim))
    V = torch.permute(V, (0, 2, 1, 3))


    scores = torch.matmul(Q, K.transpose(-2, -1)) # [B, num_heads, L, L]

    scores = scores / math.sqrt(self.head_dim)

    if casual:
      mask = torch.triu(torch.ones(Lq, Lk), diagonal=1).bool()

      scores = scores.masked_fill(mask, float('-inf'))

    out = torch.softmax(scores, dim=-1)

    final = torch.matmul(out, V) # [B, num_heads, seq_len, heads_dim]

    final = torch.permute(final, (0, 2, 1, 3)) # [B, seq_len, num_heads, heads_dim]

    final = torch.reshape(final, (B, Lq, self.model_dim)) #[B, seq_len, model_dim]

    return self.WO(final)

#att = MultiHeadAttention(516)

In [6]:
class DecoderBlock(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super().__init__()
    self.SelfAttention = SelfAttention(d_model, num_heads)
    self.CrossAttention = CrossAttention(d_model, num_heads)
    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.ln3 = nn.LayerNorm(d_model)

    self.FFN = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
    self.dropout = nn.Dropout(dropout)

  def forward(self, x_dec, x_enc):
    # in -> [B, L, d_model]
    # out -> [B, L, d_model]

    norm_x = self.ln1(x_dec)
    atrakcja = self.SelfAttention(norm_x, norm_x, norm_x, casual=True) # [B, L, d_model]
    x = x_dec + self.dropout(atrakcja)

    norm_x2 = self.ln2(x)
    atrakcja_2 = self.CrossAttention(norm_x2, x_enc)
    x = x + self.dropout(atrakcja_2)

    norm_x3 = self.ln3(x)
    ffn = self.FFN(norm_x3)
    x = x + self.dropout(ffn)

    return x


In [7]:
class EncoderBlock(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super().__init__()
    self.attention = SelfAttention(d_model, num_heads)
    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)

    self.FNN = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, mask=None):
    # in -> [B, L, d_model]
    # out -> [B, L, d_model]

    norm_x = self.ln1(x)
    atrakcja = self.attention(norm_x, norm_x, norm_x) # [B, L, d_model]
    x = x + self.dropout(atrakcja)

    norm_x2 = self.ln2(x)
    ffn_out = self.FNN(norm_x2)
    x = x + self.dropout(ffn_out)

    return x


In [ ]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, d_model)
    self.layers = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
    self.model_dim = d_model

  def forward(self, x):
    B, L = x.shape

    x = self.embed(x) + get_sinusoidal_embeddings(L, self.model_dim) # [B, L, d_model]

    for layer in self.layers:
      x = layer(x)

    return x

In [ ]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, d_model)
    self.layers = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
    self.model_dim = d_model

  def forward(self, x, enc_out):
    # in -> [B, L]
    # out -> [B, L, vocab_size]

    B, L = x.shape

    x = self.embed(x) + get_sinusoidal_embeddings(L, self.model_dim) # [B, L, d_model]

    for layer in self.layers:
      x = layer(x, enc_out)

    return x

In [ ]:
class Transformer(nn.Module):
  def __init__(self, src_vocab, tgt_vocab, d_model, num_heads, d_ff, layers):
    super().__init__()
    self.encoder = Encoder(src_vocab, d_model, num_heads, d_ff, layers)
    self.decoder = Decoder(tgt_vocab, d_model, num_heads, d_ff, layers)

    self.lin = nn.Linear(d_model, tgt_vocab)

  def forward(self, src, tgt):
    enc = self.encoder(src)
    dec = self.decoder(tgt, enc)

    return self.lin(dec) # [B, L, tgt_vocab]